In [2]:
#Menyiapkan Dataset
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/nico/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/nico/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/nico/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/nico/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, row_number, count as spark_count
from pyspark.sql.window import Window
import pandas as pd

# 1. Inisialisasi/Nyalakan kembali SparkSession
spark = SparkSession.builder \
    .appName("Tugas5") \
    .master("local[*]") \
    .getOrCreate()

# 2. Baca data transaksi dari HDFS
df_transaksi = spark.read.csv("hdfs://localhost:9000/user/nico/tugas5/transaksi_tugas5.csv", header=True, inferSchema=True)
df_transaksi = df_transaksi.withColumn("pendapatan", col("unit_terjual") * col("harga_satuan"))

# 3. Buat DataFrame target
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

print("SparkSession berhasil diaktifkan & data berhasil dimuat!")

SparkSession berhasil diaktifkan & data berhasil dimuat!


In [22]:
# Meringkas total pendapatan per kota
df_agg_kota = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Join dengan df_target dan menghitung persentase pencapaian
df_kinerja = df_agg_kota.join(df_target, on="kota", how="inner")
df_kinerja = df_kinerja.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan")) * 100
)

print("--- BAGIAN A: Kinerja Cabang & Pencapaian Target ---")
df_kinerja.orderBy(col("pencapaian_persen").desc()).show()

--- BAGIAN A: Kinerja Cabang & Pencapaian Target ---


[Stage 7:>                                                          (0 + 4) / 4]

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [23]:
# Meringkas total pendapatan per kota dan per kategori
df_kategori_kota = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan_kategori")
)

# Mendefinisikan Window Specification (di-partition berdasarkan kota)
window_spec = Window.partitionBy("kota").orderBy(col("total_pendapatan_kategori").desc())

# Mengambil peringkat 1 (top-1) menggunakan row_number()
df_kategori_terlaris = df_kategori_kota.withColumn("rn", row_number().over(window_spec)) \
    .filter(col("rn") == 1) \
    .drop("rn")

print("--- BAGIAN B: Kategori Terlaris di Setiap Kota ---")
df_kategori_terlaris.orderBy("kota").show()

--- BAGIAN B: Kategori Terlaris di Setiap Kota ---
+----------+--------------------+-------------------------+
|      kota|            kategori|total_pendapatan_kategori|
+----------+--------------------+-------------------------+
|  Magelang|Kesehatan & Kecan...|                  7275000|
| Purworejo|Kesehatan & Kecan...|                 10075000|
|  Semarang|        Rumah Tangga|                 11125000|
|      Solo|Kesehatan & Kecan...|                  8425000|
|Yogyakarta|             Fashion|                 13325000|
+----------+--------------------+-------------------------+



In [24]:
# Mendaftarkan temporary view untuk Spark SQL
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Jalankan kueri SQL murni
hasil_sql = spark.sql('''
    SELECT 
        t.kota, 
        tg.pic_cabang, 
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
''')

print("--- BAGIAN C: Jumlah Transaksi per Kota (Spark SQL) ---")
hasil_sql.show()

--- BAGIAN C: Jumlah Transaksi per Kota (Spark SQL) ---
+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+



In [25]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.


### Bagian D: Kesimpulan Kinerja Cabang

Berdasarkan hasil analisis data transaksi dan target bulanan pada Bagian A dan Bagian B, berikut adalah kesimpulan mengenai kinerja cabang e-commerce:

1. Cabang Berkinerja Paling Baik:
   Cabang yang berkinerja paling baik adalah Cabang Purworejo dengan PIC Fitri. Cabang ini mencatatkan pencapaian target tertinggi yaitu sebesar 152,17%, di mana total pendapatan yang didapatkan mencapai Rp 45.650.000 dari target bulanan sebesar Rp 30.000.000. Pendorong utama keberhasilan Cabang Purworejo adalah tingginya angka penjualan pada kategori Kesehatan & Kecantikan yang menjadi kategori terlaris dengan total pendapatan kategori sebesar Rp 10.075.000.

2. Cabang yang Paling Perlu Perhatian Manajemen:
   Cabang yang paling memerlukan perhatian khusus dari manajemen adalah Cabang Semarang dengan PIC Sari dan Cabang Magelang dengan PIC Rani. Cabang Semarang memiliki persentase pencapaian paling rendah yaitu 69,41% (pendapatan Rp 38.175.000 dari target Rp 55.000.000). Sementara itu, Cabang Magelang berada di posisi kedua terendah dengan pencapaian 70,33% (pendapatan Rp 31.650.000 dari target Rp 45.000.000). 

Rekomendasi:
Manajemen disarankan untuk mengevaluasi strategi promosi di Cabang Semarang dan Magelang. Sebagai langkah taktis, manajemen dapat memberikan program *cross-selling* atau diskon khusus pada produk selain kategori terlaris mereka untuk mendorong peningkatan volume penjualan secara keseluruhan.